# 07 · Analysis: results table, significance, figures, examples
**Purpose:** turn the saved eval/curation outputs into the headline table (accuracy, macro-F1 ± 95 % CI, format failures), paired significance (AuditKIT paired bootstrap), and figures.
**Inputs:** `results/eval_*.json`, `results/predictions_*.jsonl`, `data/*stats.json`, curator outputs. **Outputs:** `results/results_table.{csv,md}`, `results/significance.json`, `results/caught_examples.json`, `results/d_right_b_wrong.json`, `figures/*.png`.
**Runtime:** ~2 min (CPU).

In [ ]:
import os, sys, json, random
from collections import Counter
if not os.path.ismount("/content/drive"):
    from google.colab import drive; drive.mount("/content/drive")
sys.path.insert(0, "/content/drive/MyDrive/ps8/repo/src")
import ps8_common as C
cfg = C.load_config()
import pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Validated categorical palette (dataviz reference instance), fixed order by model id; text stays in ink colors.
COLORS = {"A": "#2a78d6", "B": "#eb6834", "C": "#1baf7a", "D": "#eda100", "T": "#e87ba4"}
INK, INK2, SURFACE = "#0b0b0b", "#52514e", "#fcfcfb"
NAMES = {"A": "A · base (zero-shot)", "B": "B · RAW", "C": "C · RAW-MATCHED", "D": "D · CURATED", "T": "T · teacher"}
plt.rcParams.update({"figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "axes.edgecolor": INK2, "text.color": INK,
                     "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 11})

models = [m for m in ("A", "B", "C", "D", "T") if (C.RESULTS / f"eval_{m}.json").exists()]
ev = {m: json.load(open(C.RESULTS / f"eval_{m}.json")) for m in models}
pr = {m: C.read_jsonl(C.RESULTS / f"predictions_{m}.jsonl") for m in models}
print("models:", models)

In [ ]:
rows = []
for m in models:
    r = ev[m]
    rows.append({"model": NAMES[m], "accuracy": round(r["accuracy"], 3),
                 "acc_95ci": f"[{r['accuracy_ci95'][0]:.3f}, {r['accuracy_ci95'][1]:.3f}]",
                 "macro_f1": round(r["macro_f1"], 3),
                 "mf1_95ci": f"[{r['macro_f1_ci95'][0]:.3f}, {r['macro_f1_ci95'][1]:.3f}]",
                 **{f"f1_{k}": round(v, 3) for k, v in r["f1_per_class"].items()},
                 "format_fail": round(r["format_failure_rate"], 3), "lex_groundedness": round(r["lexical_groundedness"], 3)})
table = pd.DataFrame(rows)
table.to_csv(C.RESULTS / "results_table.csv", index=False)
(C.RESULTS / "results_table.md").write_text(table.to_markdown(index=False))
print(table.to_markdown(index=False))

In [ ]:
# Paired significance on the same 500 TEST items: AuditKIT's paired bootstrap on per-item correctness (accuracy),
# plus a paired bootstrap on macro-F1 (same resampled indices for both models).
from auditkit._bootstrap import paired_bootstrap

def paired_macro_f1(a, b, n=1000, seed=42):
    gold = [x["gold"] for x in pr[a]]
    pa, pb = [x["pred"] for x in pr[a]], [x["pred"] for x in pr[b]]
    rng, idx = random.Random(seed), range(len(gold))
    obs = C.label_metrics(gold, pb)["macro_f1"] - C.label_metrics(gold, pa)["macro_f1"]
    deltas = []
    for _ in range(n):
        s = [rng.choice(idx) for _ in idx]
        g = [gold[i] for i in s]
        deltas.append(C.label_metrics(g, [pb[i] for i in s])["macro_f1"] - C.label_metrics(g, [pa[i] for i in s])["macro_f1"])
    p = (sum(d <= 0 for d in deltas) if obs > 0 else sum(d >= 0 for d in deltas)) / n * 2
    deltas.sort()
    return {"delta": round(obs, 4), "ci95": [round(deltas[int(.025 * n)], 4), round(deltas[int(.975 * n) - 1], 4)], "p_two_sided": min(1.0, p)}

sig = {}
for a, b in [("C", "D"), ("B", "D"), ("A", "D"), ("B", "C"), ("A", "B")]:
    if a in pr and b in pr:
        assert [x["pubid"] for x in pr[a]] == [x["pubid"] for x in pr[b]]
        pairs = [(float(x["pred"] == x["gold"]), float(y["pred"] == y["gold"])) for x, y in zip(pr[a], pr[b])]
        sig[f"{b}_vs_{a}"] = {"accuracy": paired_bootstrap(pairs, n_resamples=cfg["eval"]["bootstrap_resamples"], seed=cfg["seed"]),
                              "macro_f1": paired_macro_f1(a, b, cfg["eval"]["bootstrap_resamples"], cfg["seed"])}
json.dump(sig, open(C.RESULTS / "significance.json", "w"), indent=1)
print(json.dumps(sig, indent=1))

In [ ]:
# Bar chart with 95 % CIs: accuracy and macro-F1 side by side (same 0-1 scale, one axis each).
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, key, title in [(axes[0], "accuracy", "Accuracy"), (axes[1], "macro_f1", "Macro-F1 (yes/no/maybe)")]:
    vals = [ev[m][key] for m in models]
    lo = [ev[m][key] - ev[m][f"{key}_ci95"][0] for m in models]
    hi = [ev[m][f"{key}_ci95"][1] - ev[m][key] for m in models]
    ax.bar(range(len(models)), vals, color=[COLORS[m] for m in models], width=0.62, edgecolor=SURFACE, linewidth=2)
    ax.errorbar(range(len(models)), vals, yerr=[lo, hi], fmt="none", ecolor=INK2, elinewidth=1.5, capsize=4)
    for i, v in enumerate(vals):
        ax.text(i, v + hi[i] + 0.015, f"{v:.3f}", ha="center", color=INK, fontsize=10)
    ax.set_xticks(range(len(models)), [NAMES[m].replace(" · ", "\n") for m in models], fontsize=9)
    ax.set_ylim(0, 1); ax.set_title(title, color=INK, loc="left"); ax.grid(axis="y", color="#e6e5e0", linewidth=0.8)
    ax.set_axisbelow(True)
fig.suptitle("PubMedQA TEST (n=500, expert labels) — 95% bootstrap CIs", color=INK, x=0.01, ha="left")
fig.tight_layout(); fig.savefig(C.FIGURES / "results_bars.png", dpi=160); plt.close(fig)

In [ ]:
# Confusion matrices (rows = gold, cols = predicted; "invalid" = format failure).
fig, axes = plt.subplots(1, len(models), figsize=(3.3 * len(models), 3.4))
for ax, m in zip(axes, models):
    cm = ev[m]["confusion"]; mat = cm["matrix"][:3]
    ax.imshow(mat, cmap="Blues", vmin=0)
    for i, row in enumerate(mat):
        for j, v in enumerate(row):
            ax.text(j, i, v, ha="center", va="center", color=INK if v < max(map(max, mat)) * 0.6 else SURFACE, fontsize=10)
    ax.set_xticks(range(4), ["yes", "no", "maybe", "inv."], fontsize=9); ax.set_yticks(range(3), ["yes", "no", "maybe"], fontsize=9)
    ax.set_title(NAMES[m], fontsize=10, color=INK); ax.set_xlabel("predicted", fontsize=9)
axes[0].set_ylabel("gold", fontsize=9)
fig.tight_layout(); fig.savefig(C.FIGURES / "confusion_matrices.png", dpi=160); plt.close(fig)

In [ ]:
# Curation funnel: generated QA → rejected by gate → curated.
cs = json.load(open(C.DATA / "curation_stats.json"))
steps = sorted(cs["rejected_by_step"].items(), key=lambda kv: -kv[1])
labels = ["generated QA"] + [f"rejected: {k}" for k, _ in steps] + ["CURATED (passed)"]
vals = [cs["generated_qa"]] + [v for _, v in steps] + [cs["passed"]]
fig, ax = plt.subplots(figsize=(8, 0.6 * len(labels) + 1))
ax.barh(range(len(labels))[::-1], vals, color=["#2a78d6"] + ["#c3c2b7"] * len(steps) + ["#eda100"], height=0.6)
for i, v in zip(range(len(labels))[::-1], vals):
    ax.text(v + max(vals) * 0.01, i, f"{v:,}", va="center", color=INK, fontsize=10)
ax.set_yticks(range(len(labels))[::-1], labels); ax.set_xlim(0, max(vals) * 1.12)
ax.set_title(f"Curation funnel — {cs['n_source']} unlabelled abstracts → {cs['generated_qa']:,} QA → {cs['passed']:,} curated "
             f"({cs['rejection_rate']:.0%} rejected)", color=INK, loc="left", fontsize=11)
fig.tight_layout(); fig.savefig(C.FIGURES / "curation_funnel.png", dpi=160); plt.close(fig)
print(json.dumps(cs, indent=1))

In [ ]:
# Label distribution: expert TEST vs the three training sets (stacked to 100 %).
ts = json.load(open(C.DATA / "training_set_stats.json"))["sets"]
test = C.read_jsonl(C.ART / "data" / "test.jsonl")
dist = {"TEST (expert)": Counter(r["final_decision"] for r in test), "RAW": ts["raw"]["labels"],
        "RAW-MATCHED": ts["raw_matched"]["labels"], "CURATED": ts["curated"]["labels"]}
LAB_COL = {"yes": "#2a78d6", "no": "#eb6834", "maybe": "#1baf7a"}
fig, ax = plt.subplots(figsize=(8, 3.2))
for i, (name, c) in enumerate(dist.items()):
    tot, left = sum(c.get(k, 0) for k in C.LABELS), 0
    for k in C.LABELS:
        w = c.get(k, 0) / tot
        ax.barh(i, w, left=left, color=LAB_COL[k], edgecolor=SURFACE, linewidth=2, height=0.6, label=k if i == 0 else None)
        if w > 0.06:
            ax.text(left + w / 2, i, f"{k} {w:.0%}", ha="center", va="center", color=SURFACE, fontsize=9)
        left += w
ax.set_yticks(range(len(dist)), list(dist)); ax.invert_yaxis(); ax.set_xlim(0, 1)
ax.legend(ncol=3, loc="lower right", bbox_to_anchor=(1, 1.0), frameon=False)
ax.set_title("Label distribution", color=INK, loc="left")
fig.tight_layout(); fig.savefig(C.FIGURES / "label_distribution.png", dpi=160); plt.close(fig)

In [ ]:
# Examples the gates caught: one per rejecting step (seeded), with CuratorKIT's structured reason.
rej = [r for d in sorted((C.ART / "curator" / "full").glob("chunk_*")) for r in C.read_jsonl(d / "rejected.jsonl") if r["source_pubid"]]
rng = random.Random(cfg["seed"])
caught = []
for step in [k for k, _ in steps]:
    pool = [r for r in rej if r["rejecting_step"] == step and C.parse_answer(r["answer"])]
    for r in rng.sample(pool, min(2, len(pool))):
        caught.append({k: r[k] for k in ("source_pubid", "rejecting_step", "rejection_reason", "question", "answer", "context_text")})
json.dump(caught, open(C.RESULTS / "caught_examples.json", "w"), indent=1)
for r in caught:
    print(f"\n### {r['rejecting_step']} — {r['rejection_reason']}\nQ: {r['question']}\nA: {r['answer']}\nABSTRACT: {r['context_text'][:700]}…")

In [ ]:
# Items where D (curated) is right and B (raw) is wrong.
if "B" in pr and "D" in pr:
    tmap = {r["pubid"]: r for r in test}
    wins = [(b, d) for b, d in zip(pr["B"], pr["D"]) if d["pred"] == d["gold"] and b["pred"] != b["gold"]]
    losses = sum(1 for b, d in zip(pr["B"], pr["D"]) if b["pred"] == b["gold"] and d["pred"] != d["gold"])
    ex = [{"pubid": b["pubid"], "question": tmap[b["pubid"]]["question"], "gold": b["gold"], "B_output": b["output"],
           "D_output": d["output"]} for b, d in rng.sample(wins, min(3, len(wins)))]
    json.dump({"D_right_B_wrong": len(wins), "B_right_D_wrong": losses, "examples": ex},
              open(C.RESULTS / "d_right_b_wrong.json", "w"), indent=1)
    print(f"D right & B wrong: {len(wins)} | B right & D wrong: {losses}")
    for e in ex:
        print(f"\n### {e['question']} (gold={e['gold']})\nB: {e['B_output']}\nD: {e['D_output']}")

In [ ]:
print("Saved figures:", sorted(p.name for p in C.FIGURES.glob("*.png")))
print("Saved results:", sorted(p.name for p in C.RESULTS.glob("*")))